## Cricket Stats Assistant
- **Goal**: Feed it match reports or player stats and ask for comparisons.   
- **Focus**: Working with structured vs. unstructured data and specialized entity recognition.


In [2]:
import os

# Define your dependencies here
requirements = """
mistralai<2.0.0
jedi>=0.16
llama-index-llms-google-genai
llama-index-embeddings-google-genai
llama-index-embeddings-huggingface
llama-index
llama-index-llms-groq
llama-index-postprocessor-cohere-rerank
pandas
llama-index-readers-wikipedia
wikipedia
llama-index-experimental
"""

# Write the file to the local Colab disk
with open("requirements.txt", "w") as f:
    f.write(requirements.strip())

print("✅ requirements.txt has been recreated.")

# Install everything silently
!pip install -q -r requirements.txt
print("🚀 Libraries installed successfully.")

✅ requirements.txt has been recreated.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Downloading Unstructured Data
Wikipedia Reader

In [ ]:
import wikipedia
from llama_index.core import Document

# --- Set user_agent manually ---
wikipedia.set_user_agent("MyCricketProject/1.0 (contact: your-email@example.com)")

pages = ['Cricket', 'Cricket statistics', 'Virat Kohli', '2023 Cricket World Cup', 'Laws of Cricket']
documents = []

for page in pages:
    try:
        content = wikipedia.page(page, auto_suggest=False).content
        documents.append(Document(text=content, extra_info={"title": page}))
    except Exception as e:
        print(f"Error loading {page}: {e}")

# Proceed with 'docs' instead of 'documents'

In [ ]:
print(documents[1])

Doc ID: 09e07605-a23f-4128-8589-abe07c506727
Text: Cricket is a sport that generates a variety of statistics.
Statistics are recorded for each player during a match, and aggregated
over a career. At the professional level, statistics for Test cricket,
One Day International (ODI), and first-class cricket are recorded
separately. However, since Test matches are a form of first-class
cricket, a pla...


## Downloading Structured Data
From Cricsheet

In [ ]:
import requests
import zipfile
import io
import os

# --- CONFIGURATION ---
# Example: IPL JSON data.
# You can find other URLs at https://cricsheet.org/downloads/
DATA_URL = "https://cricsheet.org/downloads/ipl_json.zip"
EXTRACTION_TARGET = "cricket_data"

def download_and_extract(url, target_dir):
    print(f"🚀 Starting download from: {url}")

    # 1. Send request to Cricsheet
    response = requests.get(url)

    if response.status_code == 200:
        print("✅ Download complete. Extracting files...")

        # 2. Create directory if it doesn't exist
        if not os.path.exists(target_dir):
            os.makedirs(target_dir)

        # 3. Extract the ZIP content from memory
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            z.extractall(target_dir)

        # 4. Summary
        file_count = len(os.listdir(target_dir))
        print(f"📂 Extraction successful!")
        print(f"📍 Location: /content/{target_dir}")
        print(f"📊 Total files extracted: {file_count}")
    else:
        print(f"❌ Failed to download. Status code: {response.status_code}")

# Run the function
download_and_extract(DATA_URL, EXTRACTION_TARGET)

🚀 Starting download from: https://cricsheet.org/downloads/ipl_json.zip
✅ Download complete. Extracting files...
📂 Extraction successful!
📍 Location: /content/cricket_data
📊 Total files extracted: 1212


## Vector Store for Wiki Docs (Unstructured)
(Set Global Settings first)

In [ ]:

# from llama_index.llms.google_genai import GoogleGenAI
from llama_index.llms.groq import Groq
# from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, Settings
from google.colab import userdata
import os
import nest_asyncio

# 1. Fix the asyncio loop for Colab
nest_asyncio.apply()

# 1. Setup API Key
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# 2. Use 'llama3-70b-8192' from Groq - it's fast and smart enough for RAG
# Settings.llm = Groq(model="llama-3.3-70b-versatile")
Settings.llm = Groq(model="llama-3.1-8b-instant", temperature=0)

# 3. Setup the Embedding Model (Explicitly using Google) -- QUOTA EXCEEDING; SWITCH TO LOCAL
# 'text-embedding-004' is the current standard for high-performance RAG
# Settings.embed_model = GoogleGenAIEmbedding(model_name="models/gemini-embedding-2")

# This runs LOCALLY on your Colab machine—no API key, no 429 errors!
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Settings.chunk_size = 1024  # (reverted) Doubling default chunk size to keep the list (42 laws of cricket) together
Settings.chunk_overlap = 200 # Overlap ensures no laws get cut in half at the border

# 4. Create the Vector Index
# This chunks the text, creates embeddings, and stores them in memory
# show_progress=True helps you see it working since Wiki pages are long
vector_index = VectorStoreIndex.from_documents(documents, show_progress=True)


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/46 [00:00<?, ?it/s]

## First RAG Attempt

In [ ]:
from llama_index.core import PromptTemplate

# 1. Define your "Production-Grade" Prompt
# We use strict language to keep the 8B model on track (No knowledge leak)
strict_qa_tmpl_str = (
            "Context information is below.\n"
            "---------------------\n"
            "{context_str}\n"
            "---------------------\n"
            "Using ONLY the context above, answer: {query_str}\n"
            "If the information is not in the context, say 'I don't know based on the documents.' "
            "Do not use external knowledge."
)
qa_prompt = PromptTemplate(strict_qa_tmpl_str)

# 2. Create the Wiki Query Engine with all settings at once
wiki_query_engine = vector_index.as_query_engine(
    similarity_top_k=5, # default top_k is 2
    text_qa_template=qa_prompt  # <--- Added directly here
)

In [ ]:
# --- QUICK TEST ---
response = wiki_query_engine.query("What are the 42 laws of cricket?")
print(response)

**Rewrite**

The 42 Laws of Cricket are codified in The Laws of Cricket, which has a global remit. These laws are divided into various sections, including those that cover setting up the game, overs, scoring, dead ball and extras, players, substitutes and practice, unfair play, and players' conduct. The laws are supplemented by Appendices A-E, which provide definitions, specifications, and restrictions on equipment and player conduct.


In [ ]:
response = wiki_query_engine.query("Provide a numbered list of all 42 laws of cricket based on the documents?")
print(response)

**Rewrite**

Based on the provided context, here is the list of 42 Laws of Cricket:

1. The players. A cricket team consists of eleven players, including a captain.
2. The umpires. There are two umpires, who apply the Laws, make all necessary decisions, and relay the decisions to the scorers.
3. The scorers. There are two scorers who respond to the umpires' signals and keep the score.
4. The ball. A cricket ball is between 8.81 and 9 inches (22.4 cm and 22.9 cm) in circumference, and weighs between 5.5 and 5.75 ounces (155.9g and 163g) in men's cricket.
5. The bat. The bat is no more than 38 inches (96.52 cm) in length, no more than 4.25 inches (10.8 cm) wide, no more than 2.64 inches (6.7 cm) deep at its middle and no deeper than 1.56 inches (4.0 cm) at the edge.
6. The pitch. The pitch is a rectangular area of the ground 22 yards (20.12 m) long and 10 ft (3.05 m) wide.
7. The creases. This Law sets out the dimensions and locations of the creases.
8. The wickets. Measurements and diag

In [ ]:
# Testing faithfulness/Groundedness
response = wiki_query_engine.query("Who is Mahatma Gandhi?")
print(response)

I don't know based on the documents.


## Re-Ranking Technique
(improving accuracy of RAG response)

In [ ]:
from llama_index.postprocessor.cohere_rerank import CohereRerank

def get_wiki_query_engine(vector_index, use_reranker=False):
    """
    Returns a query engine with or without re-ranking.
    If use_reranker is True:
        - similarity_top_k = 25 (5 * 5)
        - uses CohereRerank to trim back to top 5 nodes.
    If False:
        - similarity_top_k = 5
        - no post-processing.
    """

    # Logic for top_k and post-processors
    top_k = 25 if use_reranker else 10
    postprocessors = []

    if use_reranker:
        # Initialize Cohere Reranker
        # We take the top 25 and distill them down to the best 5 for the LLM
        cohere_rerank = CohereRerank(
            api_key=userdata.get('COHERE_API_KEY'),
            top_n=10
        )
        postprocessors.append(cohere_rerank)
        print("🚀 Initializing engine WITH Cohere Re-ranker (Top-K: 25 -> 5)")
    else:
        print("⚡ Initializing baseline engine (Top-K: 5)")

    # Build the engine
    return vector_index.as_query_engine(
        similarity_top_k=top_k,
        node_postprocessors=postprocessors,
        text_qa_template=qa_prompt  # adding system prompt
    )

# --- COMPARISON TEST ---

# 1. Test without reranking
engine_off = get_wiki_query_engine(vector_index, use_reranker=False)
response_off = engine_off.query("List the 42 laws of cricket?")
print(f"\nWITHOUT RERANK:\n{response_off}\n")

# 2. Test with reranking
engine_on = get_wiki_query_engine(vector_index, use_reranker=True)
response_on = engine_on.query("List the 42 laws of cricket?")
print(f"\nWITH RERANK:\n{response_on}")

⚡ Initializing baseline engine (Top-K: 5)

WITHOUT RERANK:
**Rewrite**

The Laws of Cricket are a comprehensive set of rules governing the game, consisting of 42 laws that cover various aspects of the game. They are divided into several categories, including the equipment used, the conduct of players, and the rules of play. The Laws cover the size and weight of the ball and bat, the dimensions of the pitch and creases, and the rules for scoring runs and declaring innings. They also address issues such as unfair play, the conduct of players, and the responsibilities of umpires and scorers. The Laws are designed to ensure that the game is played fairly and safely, and to promote the spirit of sportsmanship and fair play.

🚀 Initializing engine WITH Cohere Re-ranker (Top-K: 25 -> 5)

WITH RERANK:
**Rewrite**

The Laws of Cricket are divided into several categories, including those that cover the players and officials, basic equipment, pitch specifications, and timings of play. These are f

## Debugging retrieval

In [ ]:
# Output the Global chunk size settins
print(f"Global Chunk Size: {Settings.chunk_size} tokens (not characters)") # Default is usually 1024

# Output the actual character length of the nodes retrieved
for i, node in enumerate(response.source_nodes):
    print(f"Node {i} character count: {len(node.get_content())}")

Global Chunk Size: 2048 tokens (not characters)
Node 0 character count: 8789
Node 1 character count: 7308
Node 2 character count: 5713
Node 3 character count: 6679
Node 4 character count: 5810


In [ ]:
# # Debuggin the incorrect response issue

# View the actual text chunks the LLM was given
for i, node in enumerate(response_on.source_nodes):
    print(f"--- Node {i} (Source: {node.metadata.get('title')}) ---")
    print(node.get_content()[:500] + "...") # Print first 500 chars
    print(f"Score: {node.score}\n")

--- Node 0 (Source: Laws of Cricket) ---
The Laws of Cricket is a code that specifies the rules of the game of cricket worldwide. The earliest known code was drafted in 1744. Since 1788, the code has been owned and maintained by the private Marylebone Cricket Club (MCC) in Lord's Cricket Ground, London. There are currently 42 Laws (always written with a capital "L"), which describe all aspects of how the game is to be played. MCC has re-coded the Laws six times, each with interim revisions that produce more than one edition. The most r...
Score: 0.9997749

--- Node 1 (Source: Cricket) ---
== Laws and gameplay ==

In cricket, the rules of the game are codified in The Laws of Cricket (hereinafter called "the Laws"), which has a global remit. There are 42 Laws (always written with a capital "L"). The earliest known version of the code was drafted in 1744, and since 1788, it has been owned and maintained by its custodian, the Marylebone Cricket Club (MCC) in London.


=== Playing area ===


## Structured RAG
(for the IPL csv files)

Step 1: Flatten the IPL Data

In [ ]:
import json
import pandas as pd
from pathlib import Path

def load_ipl_data(data_dir="cricket_data"):
    all_matches = []
    pathlist = Path(data_dir).glob('*.json')

    for path in pathlist:
        with open(path, 'r') as f:
            data = json.load(f)
            # Extract basic match info
            info = data.get('info', {})

            # We only want match summaries for now
            match_summary = {
                "match_id": path.stem,
                "season": info.get('season'),
                "date": info.get('dates', [None])[0],
                "team1": info.get('teams', [None, None])[0],
                "team2": info.get('teams', [None, None])[1],
                "venue": info.get('venue'),
                "winner": info.get('outcome', {}).get('winner', 'No Result'),
                "margin_runs": info.get('outcome', {}).get('by', {}).get('runs', 0),
                "margin_wickets": info.get('outcome', {}).get('by', {}).get('wickets', 0)
            }
            all_matches.append(match_summary)

    return pd.DataFrame(all_matches)

# Create the DataFrame
df_ipl = load_ipl_data()
print(f"✅ Loaded {len(df_ipl)} matches into a DataFrame.")
df_ipl.head()

✅ Loaded 1211 matches into a DataFrame.


,match_id,season,date,team1,team2,venue,winner,margin_runs,margin_wickets
0,598043,2013,2013-05-03,Kolkata Knight Riders,Rajasthan Royals,Eden Gardens,Kolkata Knight Riders,0,8
1,392235,2009,2009-05-21,Delhi Daredevils,Mumbai Indians,SuperSport Park,Delhi Daredevils,0,4
2,419136,2009/10,2010-04-02,Kings XI Punjab,Royal Challengers Bangalore,"Punjab Cricket Association Stadium, Mohali",Royal Challengers Bangalore,0,6
3,1136607,2018,2018-05-13,Mumbai Indians,Rajasthan Royals,Wankhede Stadium,Rajasthan Royals,0,7
4,598033,2013,2013-04-27,Mumbai Indians,Royal Challengers Bangalore,Wankhede Stadium,Mumbai Indians,58,0


**Step 2: Implement the Pandas Query Engine**    
Unlike the previous Vector Index query engine, this one doesn't use embeddings; it uses the LLM to write Python code.   

Why this is a major upgrade for your Assistant:
- **Deterministic Accuracy**: If the data says KKR won 40 times, the engine will return "40." A vector index would just "guess" based on match reports.
- **Interview Concept**: This is called Text-to-SQL (or in this case, Text-to-Pandas). It demonstrates that you know how to handle "Hybrid RAG"—using different tools for different data types.

In [ ]:
from llama_index.experimental.query_engine import PandasQueryEngine
from llama_index.llms.groq import Groq

# 1. Initialize the Engine
# We'll use the 70B model here if your quota has reset,
# as it's MUCH better at writing accurate Python code than the 8B.
ipl_stats_engine = PandasQueryEngine(
    df=df_ipl,
    verbose=True,
    llm=Groq(model="llama-3.3-70b-versatile", temperature=0)
)

# 2. Test a "Math" question
response = ipl_stats_engine.query("Which team has won the most matches at Eden Gardens?")
print(response)

> Pandas Instructions:
```
df[df['venue'] == 'Eden Gardens']['winner'].value_counts().idxmax()
```
> Pandas Output: Kolkata Knight Riders
Kolkata Knight Riders


## Sync to Github repo

In [ ]:
# Run this in your sync cell
commit_message = ""

!git add .
!git commit -m "{commit_message}"
!git push origin main

In [4]:
# Create Readme file
readme_content = """
# 🏏 Cricket Stats RAG Assistant
A Retrieval-Augmented Generation (RAG) application built with **LlamaIndex**.

## 🚀 Overview
This project uses a hybrid data strategy to answer complex cricket queries:
- **Unstructured Data:** Wikipedia match reports and player biographies.
- **Structured Data:** Cricsheet ball-by-ball JSON/CSV statistics.
- **Routing:** A `RouterQueryEngine` to intelligently switch between narrative and statistical engines.

## 🛠️ Setup
1. Install dependencies: `pip install -r requirements.txt`
2. Set your `GOOGLE_API_KEY` in Colab Secrets.
3. Run the main notebook to initialize the engines.
"""

with open("README.md", "w") as f:
    f.write(readme_content.strip())

print("✅ README.md created in the sidebar.")

✅ README.md created in the sidebar.


### For syncing first time

In [6]:
from google.colab import userdata
import os

# --- 1. SETTINGS ---
TOKEN = userdata.get('GITHUB_TOKEN')
USERNAME = "vdhyani96"
REPO = "cricket-rag-project"
EMAIL = "vdhyani@cs.stonybrook.edu"

# --- 2. GIT CONFIG ---
!git config --global user.email "{EMAIL}"
!git config --global user.name "{USERNAME}"

# Ensure remote is set up correctly
if not os.path.exists('.git'):
    !git init
    !git remote add origin https://{TOKEN}@github.com/{USERNAME}/{REPO}.git
else:
    !git remote set-url origin https://{TOKEN}@github.com/{USERNAME}/{REPO}.git

# --- 3. STAGE, COMMIT, AND PUSH ---
# Using '.' adds EVERYTHING (README, requirements, .gitignore)
# while respecting your .gitignore rules.
!git add .
!git commit -m "Docs: Added README and synchronized workspace"
!git branch -M main

# No more -f needed since the histories are now aligned!
!git push -u origin main

[main 602e9da] Docs: Added README and synchronized workspace
 1 file changed, 13 insertions(+)
 create mode 100644 README.md
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 779 bytes | 779.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/vdhyani96/cricket-rag-project.git
   37524d2..602e9da  main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.
